In [1]:
import copy
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
from torchvision import transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import pandas as pd
import os
from PIL import Image
import numpy as np
from tqdm.auto import tqdm  # これを追加

DATA_DIR = '/mnt/data1/gotou/projects/Medical/kaggledata'
TRAIN_IMG_DIR = os.path.join(DATA_DIR, 'train')
LABELS_CSV = os.path.join(DATA_DIR, 'train_labels.csv')
TEST_IMG_DIR = os.path.join(DATA_DIR, 'test')

# データ拡張（学習時のみ）
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class PCamDataset(Dataset):
    def __init__(self, img_dir, labels_df, transform=None):
        self.img_dir = img_dir
        self.labels = labels_df.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        img_id = self.labels.iloc[idx, 0]
        label = self.labels.iloc[idx, 1]
        img_path = os.path.join(self.img_dir, f"{img_id}.tif")
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label

labels_df = pd.read_csv(LABELS_CSV)
train_df, val_df = train_test_split(labels_df, test_size=0.1, random_state=42, stratify=labels_df['label'])

train_dataset = PCamDataset(TRAIN_IMG_DIR, train_df, train_transform)
val_dataset = PCamDataset(TRAIN_IMG_DIR, val_df, val_transform)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=4)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
resnet50 = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)

resnet50.fc = nn.Linear(resnet50.fc.in_features, 1)
model = resnet50.to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=5e-5)

In [12]:
import torch
import torch.nn as nn
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# モデル定義（学習時と同じ）
model = models.resnet50(pretrained=False)
model.fc = nn.Linear(model.fc.in_features, 1)  # 1ユニット出力
model = model.to(device)

# 重みロード
ckpt_path = "/mnt/data1/gotou/projects/Medical/kaggledata/best_model_weights.pth"
state_dict = torch.load(ckpt_path, map_location=device)
model.load_state_dict(state_dict)
model.eval()

print(f"Loaded model from {ckpt_path}")


/home/gotou/miniconda3/envs/env/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/gotou/miniconda3/envs/env/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Loaded model from /mnt/data1/gotou/projects/Medical/kaggledata/best_model_weights.pth


In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from tqdm import tqdm

# --- mean/std の定義 ---
mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(device)
std  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(device)


# --- 正規化補助関数 ---
def denormalize(x, mean, std):
    return x * std + mean

def renormalize(x_pixel, mean, std):
    return (x_pixel - mean) / std


# --- FGSM攻撃関数 ---
def fgsm_attack_improved(model, images, labels, epsilon_pixel, device,
                         mean_tensor=None, std_tensor=None, return_preds=True):
    if mean_tensor is None or std_tensor is None:
        mean_tensor_local = mean
        std_tensor_local = std
    else:
        mean_tensor_local = mean_tensor
        std_tensor_local = std_tensor

    images = images.clone().detach().to(device)
    labels = labels.clone().detach().to(device)
    images.requires_grad = True

    outputs = model(images)
    if outputs.ndim > 1 and outputs.shape[1] == 1:
        outputs = outputs.squeeze(1)
    loss = F.binary_cross_entropy_with_logits(outputs, labels.float())
    model.zero_grad()
    loss.backward()
    grad = images.grad.data
    grad_sign = grad.sign()

    if not torch.is_tensor(epsilon_pixel):
        eps_pixel_tensor = torch.tensor(epsilon_pixel, dtype=images.dtype, device=device)
    else:
        eps_pixel_tensor = epsilon_pixel.to(device).to(images.dtype)

    eps_norm = (eps_pixel_tensor / std_tensor_local).view(1, -1, 1, 1)
    adv_images = images + eps_norm * grad_sign

    adv_pixel = denormalize(adv_images, mean_tensor_local, std_tensor_local)
    adv_pixel = torch.clamp(adv_pixel, 0.0, 1.0)
    adv_images = renormalize(adv_pixel, mean_tensor_local, std_tensor_local).detach()

    if return_preds:
        with torch.no_grad():
            adv_out = model(adv_images)
            if adv_out.ndim > 1 and adv_out.shape[1] == 1:
                adv_out = adv_out.squeeze(1)
            adv_probs = torch.sigmoid(adv_out)
            adv_preds = (adv_probs > 0.5).long().cpu()
        return adv_images, adv_preds
    else:
        return adv_images


# --- FGSM評価関数 ---
def evaluate_clean_and_fgsm(
    model, val_loader, device,
    epsilon_pixel=8/255, mean=mean, std=std,
    attack_only_correct=False, max_samples=100
):
    adv_correct, adv_total = 0, 0
    orig_correct, orig_total = 0, 0
    l2_norms, linf_norms = [], []

    model.eval()
    processed = 0

    pbar = tqdm(val_loader, desc="FGSM attack", ncols=100)
    for images, labels in pbar:
        if processed >= max_samples:
            break
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        if outputs.ndim > 1 and outputs.shape[1] == 1:
            outputs = outputs.squeeze(1)
        preds = (torch.sigmoid(outputs) > 0.5).long()

        if attack_only_correct:
            correct_mask = (preds == labels.long())
            if correct_mask.sum() == 0:
                continue
            images = images[correct_mask]
            labels = labels[correct_mask]
            # ★修正：ここで正解分だけorig_totalをカウント
            orig_correct += correct_mask.sum().item()
            orig_total += correct_mask.size(0)
        else:
            orig_correct += (preds == labels.long()).sum().item()
            orig_total += labels.size(0)

        # --- FGSM攻撃の生成を関数から呼ぶ ---
        adv_images, adv_preds = fgsm_attack_improved(
            model, images, labels,
            epsilon_pixel=epsilon_pixel,
            device=device,
            mean_tensor=mean,
            std_tensor=std,
            return_preds=True
        )

        # --- 評価 ---
        adv_correct += (adv_preds == labels.cpu()).sum().item()
        adv_total += labels.size(0)

        # ★修正：差分のサイズ計算（pixel空間で）
        orig_pixel = denormalize(images, mean, std)
        adv_pixel = denormalize(adv_images, mean, std)
        diff = (adv_pixel - orig_pixel).view(adv_pixel.size(0), -1)
        l2 = diff.norm(p=2, dim=1)
        linf = diff.abs().max(dim=1)[0]

        l2_norms.extend(l2.detach().cpu().numpy().tolist())
        linf_norms.extend(linf.detach().cpu().numpy().tolist())

        processed += labels.size(0)

    orig_acc = orig_correct / orig_total * 100
    adv_acc = adv_correct / adv_total * 100
    avg_l2 = np.mean(l2_norms)
    avg_linf = np.mean(linf_norms)

    print(f"Original Accuracy (evaluated subset): {orig_acc:.2f}% ({orig_correct}/{orig_total})")
    print(f"FGSM Adversarial Accuracy: {adv_acc:.2f}% ({adv_correct}/{adv_total})")
    print(f"Average L2 norm of perturbations:  {avg_l2:.6f}")
    print(f"Average L∞ norm of perturbations:  {avg_linf:.6f}")

    return orig_acc, adv_acc, avg_l2, avg_linf


In [14]:
evaluate_clean_and_fgsm(model, val_loader, device,
                        epsilon_pixel=8/255, max_samples=100, attack_only_correct=True)

FGSM attack:   1%|▎                                                 | 4/688 [00:01<03:47,  3.01it/s]

Original Accuracy (evaluated subset): 98.44% (126/128)
FGSM Adversarial Accuracy: 55.56% (70/126)
Average L2 norm of perturbations:  11.581202
Average L∞ norm of perturbations:  0.030128


(98.4375,
 55.55555555555556,
 np.float64(11.58120178041004),
 np.float64(0.030127718808158996))